# Implementing a **Dataloader** that fetches Input-Target Data Pairs From Scratch

We know that when we type something to a chatbot it will certainly say something. The thing that we type is `input`, while what the model says is `output`.

But what we are exploring for initial LLMs is much simpler: We assume model predict the next word as an `output` instead of giving us the answer to the questions we ask as the `output`, and the final output in both cases requires iterations.

The model predict the next word, while we are going to optimize the `parameters` of the model to let the word predicted as close to the `target` word.

<div class='alert alert-block alert-success'>
We are going to make this from scratch!
</div>

## 1. Creating input-target pairs

First, let's tokenize the text!

In [1]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2") # Instantiate GPT2 tokenizer

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
enc_text = tokenizer.encode(raw_text)


Let's create two variables `x` to store input tokens and `y` to store target tokens which are inputs shifted by 1.

The `context_size` determines how many tokens are included in the input

In [2]:
context_size = 4 #length of the input
#The context_size of 4 means that the model is trained to look at a sequence of 4 words (or tokens)
#to predict the next word in the sequence.
#The input x is the first 4 tokens [1, 2, 3, 4], and the target y is the next 4 tokens [2, 3, 4, 5]

x = enc_text[:context_size]
y = enc_text[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [40, 367, 2885, 1464]
y:      [367, 2885, 1464, 1807]


Processing the inputs along with the targets, which are the inputs shifted by one position,
we can then create the `next-word prediction` tasks as
follows:

In [3]:
for i in range(1, context_size+1):
    context = enc_text[:i]
    desired = enc_text[i]

    print(context, "---->", desired)

[40] ----> 367
[40, 367] ----> 2885
[40, 367, 2885] ----> 1464
[40, 367, 2885, 1464] ----> 1807


<div class="alert alert-block alert-info">
Everything left of the arrow (---->) refers to the input an LLM would receive, and the token
ID on the right side of the arrow represents the target token ID that the LLM is supposed to
predict.
</div>

<div class="alert alert-block alert-success">
For illustration purposes, let's repeat the previous code but convert the token IDs into
text:</div>

In [4]:
for i in range (1, context_size+1):
    context = enc_text[:i]
    desired = enc_text[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))


I ---->  H
I H ----> AD
I HAD ---->  always
I HAD always ---->  thought


## 2. Implementing a Data Loader


So far, so good! But this is just the beginning. For the efficient data loader implementation, we will use PyTorch's built-in Dataset and
DataLoader classes.

We are going to use `pytorch` to return the inputs and the targets in the form of tensor, which can be thought of multidimensional arrays, similar to `numpy.array`.

In particular, we are interested in returning two tensors: an input tensor containing the
text that the LLM sees and a target tensor that includes the targets for the LLM to predict。

Firstly, let's look at the first component: `Dataset`. We define abstract characteristics of the dataset that will be used in the dataloader.

In [5]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__ (self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        #Use a sliding window to chunk the text into overlapping sequences of max_length
        for i in range(1, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:1 + max_length]
            target_chunk = token_ids[i+1 : i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
        #Return the total number of rows in the dataset

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
        # Returns a tuple: (input_ids, target_ids)
        #Return a single row from the dataset

    '''
    These two special methods / dunder methods are required
    to let your class behave like built-in Python objects
    If you define __len__, then you can call len(my_object) and it will automatically use your method.
    If you define __getitem__, then you can use my_object[0] like a list.
    '''

Nest, let's code the `DataLoader` creation tool itself: It will use the `GPTDatasetV1` to load the inputs in batches via a PyTorch DataLoader that will be as well created.

**Though it is essential to master the basics of Python like basic use of `PyTorch`, I still think it is no longer a big deal as AI can help us code anyway🤣**


In [6]:
def create_dataloader_v1(text, batch_size=4, max_length=256,
                        stride=128, shuffle=True, drop_last=True,
                        num_workers=0):
    #Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    #Create dataset
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)

    #Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle = shuffle,
        drop_last = drop_last,
        num_workers = num_workers
    )

    return dataloader

Let's test the dataloader out, with a `batch size`(how many items you take from a list at once before moving to the next group.) of 1 and a `context size` of 4.

In [7]:
import torch
print("PyTorch version:", torch.__version__)
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

first_batch = next(iter(dataloader))
print(first_batch)

PyTorch version: 2.9.0+cu126
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]
